This notebook covers experiments with different models and evaluation of their effectiveness.
The following approaches will be checked:
- Historical Averages
    - type: baseline
    - reason: the historical average will be evaluated to provide a minimal accuracy level expected from the model
- KNN
    - type: baseline
    - reason: based on the EDA results, no linearity between the models was revealed, so knn will be evaluated as an alternative approach to the historical averages, as it essentially finds the closest data points and computes their averages.
- Poisson Regression:
    - type: baseline
    - reason: this model type perfectly fits the regression problem because of the sold_quantity as a countable value.
- CatBoost - 2 stages: Classification and Regression
    - type: advanced
    - reason: because of the zero-inflated dataset, a classification approach to predict if there will be a sale potentially (yes/no) will help to a regression model to better understand where 0 value is expected from the beginning.

In [51]:
import pandas as pd
from sklearn.metrics import mean_absolute_error
import numpy as np
pd.set_option('display.max_rows', None)

In [52]:
df = pd.read_csv("interim_for_notebooks/complete_data_with_features.csv", parse_dates=['date'])
df.head(1)

,Unnamed: 0,flight_key,flight_number,origin,destination,date,month_name,year,weekday_name,is_weekend,...,price_bin,hist_avg_l1,hist_count_l1,hist_avg_l2,hist_count_l2,hist_avg_l3,hist_count_l3,hist_avg,hist_level_used,fold_num
0,6781,1cc1c7e687ebdb646528126fce678946,AB044,city_020,city_001,2025-11-14,November,2025.0,Friday,False,...,High,0.333333,12.0,0.333333,12.0,0.420091,219,0.333333,1.0,1


## Train / test df preparation
The last 2 weeks of DF are reserved to the test set.

In [53]:
test_cutoff = df['date'].max() - pd.Timedelta(weeks=2)
print(f"Train / test split by date: {test_cutoff}")
train_df = df[df['date'] <= test_cutoff]
test_df = df[df['date'] > test_cutoff]
print(f"Train DF size: {train_df.shape}")
# test_df = test_df.drop("sold_quantity", axis=1)
print(f"Test DF size: {test_df.shape}")

Train / test split by date: 2026-02-14 00:00:00
Train DF size: (41616, 32)
Test DF size: (6139, 32)


Model's accuracy will be evaluated  by 2 different approaches:
- technical:
    - MAE - mean absolute error
    - RMSE - root mean square error
- business - by comparing the predicted value vs the factual ones:
    - accurate_predictions: when an actual value equals to the predicted one
    - potential_waste: when an actual value is lower than the predicted one
    - potential_lost_sale: when an actual value is higher than the predicted one

In [54]:
# technical metrics evaluation
def evaluate_technical_metrics(df: pd.DataFrame, models_prediction_column: str):
    y_true = df["sold_quantity"]
    y_predicted = df[models_prediction_column]

    mae = mean_absolute_error(y_true, y_predicted)
    rmse = np.sqrt((y_true - y_predicted) ** 2).mean()

    print(f"MAE:  {mae:.3f}")
    print(f"RMSE: {rmse:.3f}")

In [55]:
# business metrics evaluation
def evaluate_business_metrics(df: pd.DataFrame, models_prediction_column: str):
    y_true = df["sold_quantity"]
    y_predicted = df[models_prediction_column]
    diff = y_true - y_predicted

    results = pd.DataFrame({
        "category": ["accurate_prediction", "potential_waste", "potential_lost_sale"],
        "number of records": [
            (diff == 0).sum(),
            (diff < 0).sum(),  # predicted > actual = waste
            (diff > 0).sum()   # predicted < actual = lost sale
        ],
        "share": [
            round((diff == 0).mean() * 100, 1),
            round((diff < 0).mean() * 100, 1),
            round((diff > 0).mean() * 100, 1)
        ],
        "standard deviation": [
            0,
            diff[diff < 0].mean(),  # average over-prediction
            diff[diff > 0].mean()   # average under-prediction
        ]
    })
    print(results.to_string(index=False))

## Historical averages

The historical averages are computed based on a hierarchical approach depending on the number of available records for a given level:
- Level 1: item_id - route - day_period
- Level 2: item_id - route - is_night
- Level 3: item_id - day_period

There should be at least 10 data points available in a group to compute an average for a given combination.

In [56]:
test_df = test_df.copy()
test_df['pred_baseline'] = test_df['hist_avg'].round().clip(lower=0).astype(int)

Checking the baseline model's accuracy

In [57]:
evaluate_technical_metrics(test_df, "pred_baseline")
evaluate_business_metrics(test_df, "pred_baseline")

MAE:  0.605
RMSE: 0.605
           category  number of records  share  standard deviation
accurate_prediction               2955   48.1            0.000000
    potential_waste               2309   37.6           -1.158943
potential_lost_sale                875   14.3            1.186286


## KNN - Key Nearest Neighbours

The main reason to try KNN is to narrow the specificity for the data points. Compared to the averages by hierarchical grouping, KNN allows to focus on the nearest data points (3, 5, 7) will be tried out, and then it computes the average. The main difference in the approach will be changing dynamically the threshold, so that the model can be adapted based on the business needs: to prevent wastage values or to minimize the lost sales.

The following features will be put into the model: item_id, hist_avg, route, pax_bin, day_period

The modelling process will include the following steps:
1. Categorical features preparation: ordinal, target encoding for the categorical features
2. Numerical and encoded categorical features scaling (KNN is sensitive to the scale)
3. Building the model with different hyperparameters: n_neighbors = 3, 5, 7
4. Trying out different thresholds

In [58]:
# categorical features processing
from category_encoders import TargetEncoder
from sklearn.preprocessing import OrdinalEncoder

pax_order = ['> 100', "100 - 150", "150 - 180", '180 <']
oe = OrdinalEncoder(categories=[pax_order])
train_df["pax_bin_enc"] = oe.fit_transform(train_df[["pax_bin"]])
test_df["pax_bin_enc"] = oe.transform(test_df[["pax_bin"]])

# target encoding
te = TargetEncoder(cols=["item_id", "route", "day_period"])
train_df[["item_id_enc", "route_enc", "day_period_enc"]] = te.fit_transform(train_df[["item_id", "route", "day_period"]], train_df["sold_quantity"])
test_df[["item_id_enc", "route_enc", "day_period_enc"]] = te.transform(test_df[["item_id", "route", "day_period"]], test_df["sold_quantity"])

In [59]:
# numerical features scaling
from sklearn.preprocessing import StandardScaler

features = ["hist_avg", "pax_bin_enc", "item_id_enc", "route_enc", "day_period_enc"]

scaler = StandardScaler()
X_train = scaler.fit_transform(train_df[features])
X_test = scaler.transform(test_df[features])

y_train = train_df["sold_quantity"]
y_test = test_df["sold_quantity"]

In [ ]:
# training and predicting
from sklearn.neighbors import KNeighborsRegressor

results ={}

for k in [3, 5, 7]:
    knn = KNeighborsRegressor(n_neighbors=k, metric="euclidean")
    knn.fit(X_train, y_train)
    results[k] = knn.predict(X_test)